# 03: SQL analysis

**Goal:** Answer the two research questions in SQL: how far bus and MAX ridership has recovered relative to 2019, and whether ridership has recovered faster or slower than service.

**Input:** `data/processed/trimet_monthly.csv` (from notebook 02)

**Output:** Tables in `data/processed/`. Queries in `sql/`.

**Steps:** Load the cleaned data into DuckDB, run each query from a `.sql` file, check the results, and export the tables.

## Setup and load 

The data is loaded into a table called `monthly`.

In [1]:
import duckdb

CSV_PATH = "data/processed/trimet_monthly.csv"

con = duckdb.connect()  # DuckDB database

con.execute(f"""
    CREATE OR REPLACE TABLE monthly AS
    SELECT * FROM read_csv_auto('{CSV_PATH}')
    """)

## Check the load shape and data types

In [2]:
display(con.sql("SELECT COUNT(*) AS n_rows FROM monthly").df())  # expect 506

display(
    con.sql("DESCRIBE monthly").df()
)  # expect date, varchar, varchar, varchar, double, double

con.sql(""" 
    SELECT mode, tos, COUNT(*) AS n_months, MIN(month) AS first_month, MAX(month) AS last_month
    FROM monthly
    GROUP BY mode, tos
    ORDER BY mode, tos
    """).df()  # expect table seen in notebook 02

,n_rows
0,506


,column_name,column_type,null,key,default,extra
0,month,DATE,YES,None,None,None
1,mode,VARCHAR,YES,None,None,None
2,tos,VARCHAR,YES,None,None,None
3,mode_name,VARCHAR,YES,None,None,None
4,upt,DOUBLE,YES,None,None,None
5,vrh,DOUBLE,YES,None,None,None


,mode,tos,n_months,first_month,last_month
0,DR,PT,93,2019-01-01,2026-09-01
1,DR,TN,41,2023-05-01,2026-09-01
2,DR,TX,93,2019-01-01,2026-09-01
3,LR,DO,93,2019-01-01,2026-09-01
4,MB,DO,93,2019-01-01,2026-09-01
5,YR,PT,93,2019-01-01,2026-09-01


## Base view (bus and MAX only)

This analysis covers only bus (MB) and MAX light rail (LR). Each later query will start from this view called `core` that has only those two modes. 

In [3]:
from pathlib import Path


def read_sql(file_path):
    return Path("sql", file_path).read_text()


con.execute(read_sql("00_core_view.sql"))

## Check base view

In [4]:
con.sql(read_sql("checks/00_check.sql")).df()
# expect 93 per mode for both n_rows and n_distinct_months

,mode,n_rows,n_distinct_months,first_month,last_month
0,LR,93,93,2019-01-01,2026-09-01
1,MB,93,93,2019-01-01,2026-09-01


## 2019 baselines

Two views will hold the baselines from 2019.
- `baseline_month`: values from each month in 2019
- `baseline_year`: total values from the 2019 calendar year

In [5]:
con.execute(read_sql("01a_baseline_month_2019.sql"))
con.execute(read_sql("01b_baseline_year_2019.sql"))

## Check baselines

In [6]:
display(con.sql("SELECT * FROM baseline_month ORDER BY mode, month_num").df())
display(con.sql("SELECT * FROM baseline_year ORDER BY mode").df())

display(con.sql(read_sql("checks/01_check.sql")).df())
# expect Trues (baseline month sum = baseline year)

,mode,month_num,baseline_upt,baseline_vrh
0,LR,1,3201280.0,53964.0
1,LR,2,2861440.0,49134.0
2,LR,3,3197550.0,54111.0
3,LR,4,3209820.0,53024.0
4,LR,5,3429260.0,54476.0
5,LR,6,3331050.0,52623.0
6,LR,7,3329110.0,54867.0
7,LR,8,3151630.0,51923.0
8,LR,9,3097660.0,51678.0
9,LR,10,3442530.0,54496.0


,mode,baseline_upt_year,baseline_vrh_year
0,LR,38388070.0,636933.0
1,MB,57374350.0,2084004.0


,mode,n_months,upt_match,vrh_match
0,LR,12,True,True
1,MB,12,True,True


## Monthly recovery

`monthly_recovery` will hold each month's UPT and VRH values as a share of the same month-of-the-year in the 2019 baseline.

In [7]:
con.execute(read_sql("02_monthly_recovery.sql"))

## Check monthly recovery

- Each core row should be kept exactly once after the join.
- Recovery values for 2019 should be 1.0 for both UPT and VRH.
- Nothing should be null, since the baseline month values are complete for 2019.
- Also hand check a few values: edge cases and a pandemic month. 

In [8]:
display(con.sql(read_sql("checks/02a_check.sql")).df())
con.sql(read_sql("checks/02b_check_spot.sql")).df()

,n_rows,n_core_rows,bad_2019_rows,null_rows
0,186,186,0.0,0.0


,mode,month,upt,vrh,upt_recovery,vrh_recovery
0,LR,2019-09-01,3097660.0,51678.0,1.000,1.000
1,LR,2020-04-01,1050980.0,49257.0,0.327,0.929
2,LR,2026-09-01,1880533.0,47357.0,0.607,0.916
3,MB,2019-09-01,4723280.0,170030.0,1.000,1.000
4,MB,2020-04-01,1526440.0,151215.0,0.308,0.867
5,MB,2026-09-01,3753635.0,156209.0,0.795,0.919


**Spot check results:** The checked values match the hand-computed values from the raw source file.